# Parameter-Efficient Fine-Tuning (PEFT): Fine-tune Large Language Models with Limited Resources

Complete CPU teaching example using the official Pythia-70M checkpoint. Install pinned torch 2.10.0, transformers 4.57.6, bitsandbytes 0.50.2, peft 0.21.0, accelerate 1.12.0, huggingface-hub 0.36.2 and safetensors 0.8.0. Run all cells in order. Four explicit teaching fixtures and twenty updates per adapter demonstrate mechanics, not general model quality. No account, paid API, GPU, hosted training or upload is used. The two original archive examples remain verbatim in the appendix.

In [ ]:
import os, sys, json, hashlib, tempfile, gc, platform
from pathlib import Path
import torch
import torch.nn.functional as F
import bitsandbytes as bnb
import transformers, peft
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from huggingface_hub import snapshot_download
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training, PeftModel
torch.set_num_threads(4)
torch.manual_seed(21)
cache = Path(os.environ.get('TDB_MODEL_CACHE', str(Path.home() / '.cache' / 'to-data-beyond-models')))
device = 'cpu'
print('Python:', sys.version.split()[0], 'Platform:', platform.system(), platform.machine())
print('PyTorch:', torch.__version__, 'Transformers:', transformers.__version__, 'PEFT:', peft.__version__, 'bitsandbytes:', bnb.__version__)
print('Local CPU execution; no inference service or paid API.')

In [ ]:
model_id = 'EleutherAI/pythia-70m'
revision = 'a39f36b100fe8a5377810d56c3f4789b9c53ac42'
checkpoint = Path(snapshot_download(model_id, revision=revision, cache_dir=cache, token=False,
    allow_patterns=['config.json','tokenizer.json','tokenizer_config.json','special_tokens_map.json','model.safetensors']))
weight_file = checkpoint / 'model.safetensors'
digest = hashlib.sha256(weight_file.read_bytes()).hexdigest()
assert digest == 'ebfa4e2f18696ebd83716a0d39fe2c025f2ff8483f72a83ca59c475692fc9d15'
tokenizer = AutoTokenizer.from_pretrained(checkpoint, local_files_only=True, trust_remote_code=False)
tokenizer.pad_token = tokenizer.eos_token
print('Official model:', model_id, 'Revision:', revision)
print('Safetensors SHA256:', digest)

In [ ]:
# Four explicit teaching fixtures; not a benchmark dataset.
texts = [
    'Question: What color is a clear daytime sky? Answer: blue.',
    'Question: What is two plus three? Answer: five.',
    'Question: What animal says meow? Answer: a cat.',
    'Question: What do plants need to grow? Answer: water and light.'
]
batch = tokenizer(texts, padding=True, truncation=True, max_length=64, return_tensors='pt')
labels = batch['input_ids'].clone()
labels[batch['attention_mask'] == 0] = -100
batch['labels'] = labels
assert batch['input_ids'].shape[0] == 4 and torch.all(labels[batch['attention_mask']==0] == -100)
temporary = tempfile.TemporaryDirectory(prefix='tdb-peft-')
artifacts = Path(temporary.name)
def fingerprint(model):
    h = hashlib.sha256()
    for name, value in sorted(model.state_dict().items()):
        if 'lora_' not in name:
            h.update(name.encode()); h.update(value.detach().cpu().contiguous().reshape(-1).view(torch.uint8).numpy().tobytes())
    return h.hexdigest()
def evaluate_loss(model):
    model.eval()
    with torch.no_grad(): return float(model(**batch).loss)
def train_adapter(model):
    frozen_before = fingerprint(model)
    initial = evaluate_loss(model)
    trainable = {n:p for n,p in model.named_parameters() if p.requires_grad}
    assert trainable and all('lora_' in n for n in trainable)
    before = {n:p.detach().clone() for n,p in trainable.items()}
    optimizer = torch.optim.AdamW(list(trainable.values()), lr=0.002, weight_decay=0.0)
    losses = []; seen_nonzero = set()
    model.train()
    for step in range(20):
        optimizer.zero_grad(set_to_none=True)
        output = model(**batch)
        independent = F.cross_entropy(output.logits[:,:-1,:].contiguous().view(-1,output.logits.shape[-1]),
            batch['labels'][:,1:].contiguous().view(-1), ignore_index=-100)
        torch.testing.assert_close(output.loss, independent, atol=1e-5, rtol=1e-5)
        assert torch.isfinite(output.loss)
        output.loss.backward()
        for name, parameter in trainable.items():
            assert parameter.grad is not None and torch.isfinite(parameter.grad).all()
            if parameter.grad.abs().max() > 0: seen_nonzero.add(name)
        assert all(p.grad is None for p in model.parameters() if not p.requires_grad)
        optimizer.step(); losses.append(float(output.loss.detach()))
    final = evaluate_loss(model)
    assert final < initial and fingerprint(model) == frozen_before
    assert set(trainable) == seen_nonzero
    assert all(not torch.equal(before[n],p.detach()) for n,p in trainable.items())
    print(json.dumps({'steps':len(losses),'training_rows':len(texts),'initial_eval_loss':round(initial,6),
        'final_eval_loss':round(final,6),'all_adapter_gradients_finite':True,'base_state_unchanged':True},sort_keys=True))
    return {'initial':initial,'final':final,'steps':len(losses),'base_hash':frozen_before}
print('Teaching fixtures:', len(texts), 'Padded token shape:', list(batch['input_ids'].shape))

In [ ]:
base = AutoModelForCausalLM.from_pretrained(checkpoint, dtype=torch.float32,
    attn_implementation='eager', local_files_only=True, trust_remote_code=False)
config = LoraConfig(task_type=TaskType.CAUSAL_LM, inference_mode=False, r=32,
    lora_alpha=16, lora_dropout=0.1, target_modules=['query_key_value'])
lora = get_peft_model(base, config)
targets = [m for m in lora.modules() if hasattr(m,'lora_A')]
expected_trainable = sum(config.r * (m.in_features + m.out_features) for m in targets)
assert len(targets) == 6 and sum(p.numel() for p in lora.parameters() if p.requires_grad) == expected_trainable
lora.print_trainable_parameters()
print('Target matrices:',len(targets),'Independent r*(in+out) count:',expected_trainable)

In [ ]:
lora_metrics = train_adapter(lora)

In [ ]:
lora.eval()
with torch.no_grad(): expected_logits = lora(**batch).logits.detach().clone()
lora.save_pretrained(artifacts/'lora', safe_serialization=True)
restored_base = AutoModelForCausalLM.from_pretrained(checkpoint, dtype=torch.float32,
    attn_implementation='eager', local_files_only=True, trust_remote_code=False)
restored = PeftModel.from_pretrained(restored_base, artifacts/'lora', torch_device='cpu', device_map={'':'cpu'}).eval()
with torch.no_grad(): restored_logits = restored(**batch).logits
torch.testing.assert_close(restored_logits,expected_logits,atol=1e-5,rtol=1e-5)
merged = restored.merge_and_unload().eval()
with torch.no_grad(): merged_logits = merged(**batch).logits
torch.testing.assert_close(merged_logits,expected_logits,atol=2e-4,rtol=2e-4)
print('LoRA adapter save/reload and merged-model logits match.')
del restored, restored_base, merged, base, lora, expected_logits, restored_logits, merged_logits
gc.collect()
print("Released the unquantized teaching model.")

In [ ]:
torch.manual_seed(21)
weights = torch.randn(64,64)
packed, quant_state = bnb.functional.quantize_4bit(weights, quant_type='nf4', compress_statistics=True)
decoded = bnb.functional.dequantize_4bit(packed, quant_state)
assert packed.dtype == torch.uint8 and quant_state.quant_type == 'nf4' and quant_state.nested
assert packed.numel()*packed.element_size() < weights.numel()*weights.element_size()
assert torch.isfinite(decoded).all() and 0 < torch.mean((decoded-weights)**2) < 0.05
layer = bnb.nn.Linear4bit(64,64,bias=False,compute_dtype=torch.float32,quant_type='nf4',compress_statistics=True)
layer.load_state_dict({'weight':weights}); layer = layer.to(device)
probe = torch.randn(3,64,requires_grad=True)
actual = layer(probe)
reference = F.linear(probe, bnb.functional.dequantize_4bit(layer.weight.data,layer.weight.quant_state))
torch.testing.assert_close(actual,reference,atol=1e-5,rtol=1e-5)
actual.square().mean().backward()
assert probe.grad is not None and torch.isfinite(probe.grad).all()
print('Actual NF4 packing, nested scale quantization, dequantized linear parity and backward gradients pass.')
print('Weight-only packed bytes:', packed.numel()*packed.element_size(), 'Original float32 bytes:', weights.numel()*weights.element_size())

In [ ]:
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float32)
quantized_base = AutoModelForCausalLM.from_pretrained(checkpoint, quantization_config=bnb_config,
    dtype=torch.float32, device_map={'':'cpu'}, attn_implementation='eager',
    local_files_only=True, trust_remote_code=False)
quantized_base = prepare_model_for_kbit_training(quantized_base, use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant':False})
config = LoraConfig(r=8,lora_alpha=32,target_modules=['query_key_value'],lora_dropout=0.05,
    bias='none',task_type=TaskType.CAUSAL_LM)
qlora = get_peft_model(quantized_base,config)
qlora.config.use_cache = False
quantized_layers = [m for m in qlora.modules() if isinstance(m,bnb.nn.Linear4bit)]
assert quantized_layers and all(m.weight.quant_state.quant_type=='nf4' and m.weight.quant_state.nested for m in quantized_layers)
targets = [m for m in qlora.modules() if hasattr(m,'lora_A')]
assert sum(p.numel()for p in qlora.parameters()if p.requires_grad) == sum(config.r*(m.in_features+m.out_features)for m in targets)
qlora.print_trainable_parameters()
print('Actual NF4 layers with nested scale quantization:', len(quantized_layers))

In [ ]:
qlora_metrics = train_adapter(qlora)

In [ ]:
qlora.eval()
with torch.no_grad(): expected_logits = qlora(**batch).logits.detach().clone()
qlora.save_pretrained(artifacts/'qlora',safe_serialization=True)
restored_quantized = AutoModelForCausalLM.from_pretrained(checkpoint,quantization_config=bnb_config,
    dtype=torch.float32,device_map={'':'cpu'},attn_implementation='eager',local_files_only=True,trust_remote_code=False)
restored_quantized = prepare_model_for_kbit_training(restored_quantized,use_gradient_checkpointing=False)
restored_qlora = PeftModel.from_pretrained(restored_quantized,artifacts/'qlora',torch_device='cpu',device_map={'':'cpu'}).eval()
with torch.no_grad(): actual_logits = restored_qlora(**batch).logits
torch.testing.assert_close(actual_logits,expected_logits,atol=1e-5,rtol=1e-5)
prompt = tokenizer('Question: What color is a clear daytime sky? Answer:',return_tensors='pt')
with torch.no_grad(): generated = restored_qlora.generate(**prompt,max_new_tokens=20,do_sample=False,
    pad_token_id=tokenizer.eos_token_id,eos_token_id=tokenizer.eos_token_id,use_cache=True)
assert generated.shape[1] > prompt['input_ids'].shape[1]
assert torch.equal(generated[:,:prompt['input_ids'].shape[1]],prompt['input_ids'])
print('QLoRA adapter save/reload logits match. Actual continuation:')
print(tokenizer.decode(generated[0,prompt['input_ids'].shape[1]:],skip_special_tokens=True))

In [ ]:
assert lora_metrics['steps']==qlora_metrics['steps']==20
assert lora_metrics['final']<lora_metrics['initial'] and qlora_metrics['final']<qlora_metrics['initial']
for name in ['lora','qlora']:
    config_json=json.loads((artifacts/name/'adapter_config.json').read_text())
    assert config_json['peft_type']=='LORA' and config_json['target_modules']==['query_key_value']
    assert (artifacts/name/'adapter_model.safetensors').stat().st_size > 0
print('PASS: both real 20-step training runs, frozen base states, finite/nonzero adapter gradients, independent losses/counts, NF4 packing, persistence, merge parity and generation.')
temporary.cleanup()

## Original source-code appendix

Verbatim archived examples; do not execute these incomplete/hardware-specific blocks.

```python
from transformers import AutoModelForCausalLM
from peft import get_peft_config, get_peft_model, LoraConfig, TaskType

model = AutoModelForCausalLM.from_pretrained(model_name_or_path, device_map="auto", trust_remote_code=True) # load the model

peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM, inference_mode=False, r=32, lora_alpha=16, lora_dropout=0.1,
    target_modules=['query_key_value'] # optional, you can target specific layers using this
) # create LoRA config for the finetuning


model = get_peft_model(model, peft_config) # create a model ready for LoRA finetuning

model.print_trainable_parameters() 
# trainable params: 9,437,184 || all params: 6,931,162,432 || trainable%: 0.13615586263611604
```

```python
import torch
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "EleutherAI/gpt-neox-20b"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
) # setup bits and bytes config

model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map={"":0})

model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model) # prepares the whole model for kbit training

config = LoraConfig(
    r=8, 
    lora_alpha=32, 
    target_modules=["query_key_value"], 
    lora_dropout=0.05, 
    bias="none", 
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, config) # Now you get a model ready for QLoRA training
```